# Product Information Management with Bluestone PIM

1. Create attribute definition
1. Load data (Import API)
2. connect DAM?
3. replace service
4. test ui

Product Information Management (PIM) applications help retailers and manufacturers manage and organize their product data in a centralized and structured manner. It enables them to collect, store, enrich, and distribute accurate and consistent product information across various channels, such as e-commerce platforms, marketplaces, print catalogs, and point of sale systems. Product Information Management is an important component for retailers, and its main benefits include:

* Improved data accuracy and consistency: PIM systems ensure that all product information is standardized and up-to-date, reducing errors and inconsistencies across different channels.
* Enhanced product data quality: By providing a single source of truth for product information, a PIM system enables retailers to enrich their product data with detailed descriptions, high-quality images, and videos, which can help customers make informed purchasing decisions.
* Increased efficiency: A PIM system automates many of the manual processes involved in managing product data, freeing up staff to focus on more strategic tasks.
* Faster time-to-market: With a PIM system, retailers can quickly and easily launch new products or update existing ones, allowing them to respond faster to changing market conditions and capitalize on trends.
* Improved customer experience: Consistent and compelling product information across all channels helps create a positive shopping experience for customers, leading to increased loyalty and repeat business.

Bluestone PIM is a MACH-Based PIM for enterprise clients designed specifically for composable commerce, providing flexibility across all sales channels. Bluestone PIM is a Product Information Management solution for global retailers, manufacturers, and distributors who want to make their enterprises innovative and agile. 

In this workshop you will integrate Bluestone PIM with the Retail Demo Store. You'll create the products catalogue, publish it, and consume events from Bluestone PIM. You'll also modify the **Product** microservice to retrieve all product details from Bluestone PIM.

### Prerequisites 
In order to complete this module, you'll need:

* An instance of Retail Demo Store running in your AWS account
* A Bluestone PIM test account. If you don't have one, please [get in touch](https://www.bluestonepim.com/). *(TODO: Check preferred option)*

# Load Retail Demo Store Assets

The first step is to prepare and upload the product catalogue assets. When working with assets, there are three options:
* You can use your assets available through an existing Content Delivery Network (CDN),such as amazon CloudFront
* You can use an external Digital Assets Management (DAM) system
* You can make use of the integrated, headless DAM within Bluestone PIM. 

For the purpose of this workshop, we'll use the Bluestone PIM integrated DAM.

In order to upload the Retail Demo Store assets to Bluestone PIM, we'll need to get the assets from the Retail Demo Store Web UI S3 bucket. Go to AWS Management Console, open the AWS CloudFormation console, and then Stacks. Make sure the **View nested** option is enabled. In the list of stacks, look for a stack called *<stack-name>-Base-<ID>*, e.g. *retaildemostore-Base-DFZ5BMH13QUG*. In the Output tab, look for the **WebUIBucketName** output and copy its value.

![images/StackOutput.png](images/StackOutput.png)

The Web UI S3 bucket contains all the products images. Let's copy them locally.

In [ ]:
# Create an environment variable to store the bucket name
# NOTE: replace "retaildemostore-base-xxxxxxxxxxxx-xxx-webuibucket-xxxxxxxxxxxxx" with your bucket name
export WEB_UI_BUCKET_NAME=retaildemostore-base-xxxxxxxxxxxx-xxx-webuibucket-xxxxxxxxxxxxx
mkdir ./catalogue
aws s3 cp --recursive s3://${WEB_UI_BUCKET_NAME}/images/* ./catalogue

Now that we have the files stored locally, we can invoke the Bluestone PIM Management API to create the assets and upload the images. Creating a new asset is a multi-step process:
* First, we need to create a new empty asset.
* Then, we'll upload the asset file to an Amazon S3 bucket via a pre-signed URL.
* Then, we need to add metadata to the asset.
* Lastly, we need to complete the asset. Please note that the asset will not be available in the DAM until all operations are completed.

In [ ]:
# Fetch the Bluestone PIM Public API key from the environment variable
BLUESTONE_PIM_MGT_API_AUTH_TOKEN = '<REPLACE_WITH_YOUR_MANAGEMENT_API_AUTH_TOKEN>'
BLUESTONE_MGT_API_BASE_URL = 'https://api.test.bluestonepim.com'

headers = {
    "accept": "application/json",
    "content-type": "application/json",
    "authorization": f'Bearer {BLUESTONE_PIM_MGT_API_AUTH_TOKEN}'
}

In [ ]:
import yaml
import requests
import os

# Bluestone PIM Management API Test Endpoints
ea_url = f'{BLUESTONE_MGT_API_BASE_URL}/media/upload'
ma_url = f'{BLUESTONE_MGT_API_BASE_URL}/media/upload/{id}'
md_url = f'{BLUESTONE_MGT_API_BASE_URL}/media/upload/media/upload/{id}/done'

# iterate through each folder and files in ./catalogue
for root, dirs, files in os.walk('./catalogue'):
    for asset in files:
        # Create empy asset
        ea_payload = {
            "fileName": asset,
            "contentType": "image/jpeg"
        }
        response = requests.post(ea_url, json=ea_payload, headers=headers)
        print(response.text)

        # Upload asset to s3
        asset_id = response.json()['id']
        s3_url = response.json()['actionUrl']

        # Add asset metadata
        ma_payload = {
            "name": "Sample asset name",
            "number": "asset-123",
            "description": "More sample text about the asset",
            "labels": [
                "6c159583-a699-4f79-87bc-1852a132694c",
                "11afedc6-abc3-4f26-904a-6768e98d538c"
            ],
            "folderId": null,
            "internal": false
        }
        response = requests.post(ma_url, json=ma_payload, headers=headers)
        print(response.text)

        # Mark asset as uploaded
        response = requests.post(md_url, headers=headers)
        print(response.text)

# Load Product Catalogue
Now that we have our product images ready in the Bluestone PIM DAM, it's time to create our product catalogue. For Retail Demo Store, the product catalogue is stored in two YAML files, `categories.yaml` and `products.yaml`. You can find these two files in the `data` folder; feel free to explore the files before proceeding.

Let's first create the structure for our catalogue: we'll create the catalogue root node, then parse the `categories.yaml` file, and, for each category, we'll create a category node under root using the Bluestone PIM Management API.

In [ ]:
import yaml
import requests

# Bluestone PIM Management API Test Endpoints
cn_url = f'{BLUESTONE_MGT_API_BASE_URL}/pim/catalogs/nodes?validation=NAME'
caa_url = f'{BLUESTONE_MGT_API_BASE_URL}/pim/catalogs/nodes/{id}/assets/{assetId}'

# Create root node
payload = {
    "name": "catalog",
}
response = requests.post(cn_url, json=payload, headers=headers)

# Parse yaml file
with open("./data/categories.yaml", "r") as f:
    data = yaml.safe_load(f)
    
# Create categories nodes under root
for item in data:
    # Create category node
    cn_payload = {
        "name": item["name"],
        "number": item["id"],
        "parentId": "catalog"
    }
    response = requests.post(cn_url, json=cn_payload, headers=headers)
    print(response.text)

    # replace ID and assetId in url
    caa_url = replace(item["image"])
    response = requests.post(caa_url, headers=headers)
    print(response.text)

Now, let's load our products into the catalogue. Similarly to what we have done above, we'll parse the `products.yaml` file, and, for each item, we'll create a product node under the right category node using the Bluestone PIM Management API.

In [ ]:
import yaml
import requests

# Bluestone PIM Management API Test Endpoints
p_url = f'{BLUESTONE_MGT_API_BASE_URL}/pim/products?validation=NAME'
paa_url = f'{BLUESTONE_MGT_API_BASE_URL}/pim/products/{id}/assets/{assetId}'

# Parse yaml file
with open("./data/products.yaml", "r") as f:
    data = yaml.safe_load(f)
    
# Create categories nodes under root
for item in data:
    # Create category node
    p_payload = {
        "name": item["name"],
        "number": item["id"],
        "parentId": "catalog"
    }
    response = requests.post(p_url, json=p_payload, headers=headers)
    print(response.text)

    # replace ID and assetId in url
    paa_url = replace(item["image"])
    response = requests.post(paa_url, headers=headers)
    print(response.text)

Now that we have created the product catalogue, it is ready to be published. Before we do that, let's set up event notifications so we will be notififed once the publishing process is complete. 

# Event Notifications Set Up
In order to receive notifications from Bluestone PIM, we need to create the incoming webhook and the event processing logic.

![images/EventsNotifications.png](images/EventsNotifications.png)

Follow these steps:
1. [Deploy](https://us-east-1.console.aws.amazon.com/lambda/home#/create/app?applicationId=arn:aws:serverlessrepo:us-east-1:721177882564:applications/generic-webhook-to-eventbridge) the *'Generic webhook to EventBridge'* App from the Serverless Application Repository (SAR). Use **retaildemostore** for the `EventBusName` parameter and **BluestonePIM** for the `EventSource` parameter. Make a note of the **WebhookApiUrl** output value as we'll need this later.
2. Follow the instructions to deploy the [Amazon EventBridge to Amazon SNS](https://github.com/aws-samples/serverless-patterns/tree/main/eventbridge-sns) serverless pattern. Before proceeding to step 3., make sure to modify EventRule in the `template.yaml` file. On [line 20](https://github.com/aws-samples/serverless-patterns/blob/main/eventbridge-sns/template.yaml#L20) replace `"demo.cli"` with `"BluestonePIM"`. Make a note of the **MySnsTopicArn** output value.
3. [Subscribe](https://docs.aws.amazon.com/sns/latest/dg/sns-email-notifications.html) your email address to the **MySnsTopicArn** SNS topic.

Now that we have the incoming webhook deployed, let's go and create the webhook and events subscription on the Bluestone side. For a list of event types supported by Bluestone PIM, see the [documentation](https://help.bluestonepim.com/work-with-events#event-types). Make sure to replace the Webhook URL with the **WebhookApiUrl** output value from above.

In [ ]:
import yaml
import requests

# Bluestone PIM Management API Test Endpoints
cw_url = f'{BLUESTONE_MGT_API_BASE_URL}/notification-external/webhooks'
se_url = f'{BLUESTONE_MGT_API_BASE_URL}/notification-external/subscriptions/webhook/{webhookId}/events'

headers = {
    "accept": "application/json",
    "content-type": "application/json",
    "api-key": "lyXinh2n20TedIGPkbQ5/QZ/WqKfBnP2jD+gOHbAhgBvzOOZ6FZm9JfdJJ97n4fb"
}

# Create webhook
cw_payload = {
    "active": True,
    "secret": "123",
    "url": "REPLACE_WITH_WEBHOOK_API_URL"
}
response = requests.post(cw_url, json=cw_payload, headers=headers)
print(response.text)

# Subscribe webhook to events
se_payload = { 
    "eventTypes": ["PRODUCT_SYNC_DONE"] 
}
se_url = replace(response.headers["resource-id"])
response = requests.post(se_url, json=se_payload, headers=headers)
print(response.text)

# Publish Product Catalogue
It's time now to publish our product catalogue. When products are initially created, they are created as drafts, so the first step is to change the product state. We then need to initiate the synchronization process, and finally publish the products. When products are published, they become available via the Blustone Public API almost immediately. We'll also receive an email notification once the process is complete.

In [ ]:
import yaml
import requests

# Bluestone PIM Management API Test Endpoints
cs_url = f'{BLUESTONE_MGT_API_BASE_URL}/pim/products/states/by-ids'
ss_url = f'{BLUESTONE_MGT_API_BASE_URL}/sync/syncs'
pp_url = f'{BLUESTONE_MGT_API_BASE_URL}/sync/syncs/{syncId}/publish'

headers = {
    "accept": "application/json",
    "content-type": "application/json",
    "api-key": "lyXinh2n20TedIGPkbQ5/QZ/WqKfBnP2jD+gOHbAhgBvzOOZ6FZm9JfdJJ97n4fb"
}

# Get product IDs

# Change state
cs_payload = {
    "ids": ["1", "2"],
    "action": "PUBLISH"
}
response = requests.post(cs_url, json=cs_payload, headers=headers)
print(response.text)

# Start synchronization process
response = requests.post(ss_url, headers=headers)
print(response.text)
#{
#   "state": "REPORT_IN_PROGRESS",
#   "startedAt": "2023-10-13T11:18:50.161Z",
#   "initiatedBy": "e5ced5df-c574-4754-9ceb-153d748b549d",
#   "syncs": [
#     {
#       "id": "6529279a251f34215d62e89b",
#       "startedAt": "2023-10-13T11:18:50.161Z",
#       "updatedAt": "2023-10-13T11:18:50.161Z",
#       "initiatedBy": "e5ced5df-c574-4754-9ceb-153d748b549d",
#       "phase": "PENDING"
#     }
#   ]
# }

# Publish products
pp_url = replace(response.syncs[0]["id"])
response = requests.post(pp_url, headers=headers)
print(response.text)

# Switch from Retail Demo Store Products service to Bluestone PIM

Now that the product catalogue is available in Bluestone PIM, we need to switch from the original Retail Demo Store **Products** microservice to the Bluestone PIM Public APIs. 

![images/APIGatewayIntegration.png](images/APIGatewayIntegration.png)

## Create the integration Lambda function

We'll use a Lambda function to map the Retail Demo Store APIs to the Bluestone PIM Public APIs. In particular, these are the mappings required:

```
GET /products/id/{id} -> GET /products/{id}
GET /products/featured -> POST /products/list
GET /products/category/{name} -> GET /categories/{categoryId}/products
GET /categories/all -> GET /categories
```

First, let's [create the Lambda function](https://docs.aws.amazon.com/lambda/latest/dg/getting-started.html) using the following code:

In [ ]:
import json
import os
import requests

# Fetch the Bluestone PIM Public API key from the environment variable
BLUESTONE_PIM_PUBLIC_API_KEY = os.environ.get('BLUESTONE_PIM_PUBLIC_API_KEY')
BLUESTONE_PUBLIC_API_BASE_URL = 'https://api.test.bluestonepim.com/v1'

headers = {
    "accept": "application/json",
    "x-api-key": BLUESTONE_PIM_PUBLIC_API_KEY
}

def lambda_handler(event, context):
    # Extract the HTTP method and path from the incoming request
    http_method = event['httpMethod']
    path = event['path']

    # Map the incoming request to the corresponding Bluestone PIM API endpoint
    if http_method == 'GET' and path == '/products/id/{id}':
        product_id = event['pathParameters']['id']
        api_endpoint = f'{BLUESTONE_PUBLIC_API_BASE_URL}/products/{product_id}'
    elif http_method == 'GET' and path == '/products/featured':
        api_endpoint = f'{BLUESTONE_PUBLIC_API_BASE_URL}/products/list'
        http_method = 'POST'
    elif http_method == 'GET' and path == '/products/category/{name}':
        category_name = event['pathParameters']['name']
        category_id = get_category_id(category_name)
        api_endpoint = f'{BLUESTONE_PUBLIC_API_BASE_URL}/categories/{category_id}/products'
    elif http_method == 'GET' and path == '/categories/all':
        api_endpoint = f'{BLUESTONE_PUBLIC_API_BASE_URL}/categories'
    else:
        return {
            'statusCode': 404,
            'body': json.dumps('Not Found')
        }

    # Call the Bluestone PIM API with the mapped endpoint and method
    try:
        response = requests.request(http_method, api_endpoint, headers=headers)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return {
            'statusCode': 500,
            'body': json.dumps(f'Error calling third-party API: {str(e)}')
        }

    # Return the response from the Bluestone PIM API
    return {
        'statusCode': response.status_code,
        'headers': {
            'Content-Type': response.headers.get('Content-Type', 'application/json')
        },
        'body': response.text
    }

def get_category_id(category_name):
    # Construct the API endpoint URL
    categories_url = f'{BLUESTONE_PUBLIC_API_BASE_URL}/categories?name={category_name.lower()}"

    try:
        # Make a GET request to the API endpoint
        response = requests.get(url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching category ID: {str(e)}")
        return None

    # Parse the response JSON
    data = response.json()

    # Check if the response contains a list of categories
    if isinstance(data, list) and data:
        # Return the ID of the first category in the list
        return data[0]["id"]
    else:
        print("No categories found for the given name.")
        return None

Make sure to [configure an environment variable](https://docs.aws.amazon.com/lambda/latest/dg/configuration-envvars.html) for your Lambda function called `BLUESTONE_PIM_PUBLIC_API_KEY` and set its value to the provided Bluestone PIM Public API key. 

> **NOTE:** The Public API Key differs from the Management API key used previously. 

## Create the API Gateway integration

Go to AWS Management Console, open the AWS CloudFormation console, and then Stacks. Make sure the **View nested** option is enabled. In the list of stacks, look for a stack called *<stack-name>-ApiGateway-<ID>*, e.g. *retaildemostore-ApiGateway-Z9CZ1FUGFE0C*. In the Output tab, look for the **ApiGatewayId** output and copy its value.

![images/APIGatewayStackOutput.png](images/APIGatewayStackOutput.png)

Next, navigate to the API Gateway console, select the Retail Demo Store API Gateway using the ID you copied above. From the left menu, select **Integrations** and then **Manage integrations**; click on **Create**. On the **Create an integration**, do not attach this integration to any route at this time. Select **Lambda function** from the **integration type** drop down and then select the Lambda function created earlier in the **Integration details** section. Make sure the **Grant API Gateway permission to invoke your Lambda function** option is enabled and click on **Create**.

![images/APIGatewayLambdaIntegration.png](images/APIGatewayLambdaIntegration.png)

On the **Integration details** page, note down the **Integration ID**.

![images/APIGatewayLambdaIntegrationID.png](images/APIGatewayLambdaIntegrationID.png)

## Modify the API Gateway routes

Select the **Attach inegrations to routes** tab and select the `GET /categories/all` routes from the list. Click on the *Detach integration* button on the right.

![images/APIGatewayDetachIntegration1.png](images/APIGatewayDetachIntegration1.png)

Select the Lambda integration you created earlier from the drop down list and then click on *Attach integration*.

![images/APIGatewayDetachIntegration2.png](images/APIGatewayDetachIntegration2.png)

Repeat the same steps for the `GET /products/id/{id}`, `GET /products/featured`, and `GET /products/category/{name}` routes. 

## Test the integration from the Retail Demo Store UI

Now, navigate to the Retail Demo Store web page and browse through the products and collections. You should be able to see all product details and information, now powered by Bluestone PIM.